# Volatility Dynamics 17 — Conditional variance tuning for Student-t return Markov dynamics

**Status: registered historical exploration; no real-market run yet.** Research plan: [VD 034](../../discussions/Volatility_Dynamics/034_chatgpt_vd17_conditional_variance_preregistration.md).

Seven horizons: **5, 10, 15, 30, 60, 120 and 240 minutes**; 60m primary. This experiment **does not change Student-t return transitions, the two-factor Markov volatility transition matrix or the global VD16 tail shape**. It tunes conditional forecast variance with either current causal state information, M2's independently TRAIN-fitted variance scale, or both. The latter is a **hybrid forecast**, not a pure Markov process. Output scale overlays need not be cross-horizon path-consistent.

The historical TEST segment was already viewed in VD13–16; nothing here is prospective confirmation or a trading signal.

## 0. Environment and frozen protocol

In [1]:
from pathlib import Path
import os,sys,json,time
import numpy as np
import pandas as pd
from IPython.display import display
from dotenv import load_dotenv
ROOT=Path.cwd().resolve()
while ROOT!=ROOT.parent and not (ROOT/'src'/'data_engineering').exists():ROOT=ROOT.parent
assert (ROOT/'src'/'data_engineering').exists(), 'Run from project repository'
sys.path.insert(0,str(ROOT/'src'))
load_dotenv(ROOT/'.env')
from data_engineering.modeling import markov_lift_distributions as v14
from data_engineering.modeling import markov_conditional_variance as v17
DATA=os.getenv('DERIBIT_DATA_ROOT')
assert DATA and Path(DATA).exists(), 'Set DERIBIT_DATA_ROOT in .env'
HIST=Path(DATA)/'history'/'price_5m'
assert HIST.is_dir(), 'Missing history/price_5m folder'
OUT=ROOT/'notebooks'/'Volatility_Dynamics'/'17'
OUT.mkdir(exist_ok=True,parents=True)
ASSETS=('AVAX','BTC','ETH','HYPE','SOL','TRX','XRP')
assert v17.PARTICLES==256 and v17.BOOT==1000 and v17.CAL_LIMIT==512
start=time.monotonic()
print(json.dumps(v17.registration(),indent=2,default=str))

{
  "version": "VD17-0.1.0",
  "horizons": [
    5,
    10,
    15,
    30,
    60,
    120,
    240
  ],
  "primary_horizon": 60,
  "source": "VD14 state and TRAIN two-EMA variance, VD15 5m Student-t return kernel, VD16 shape/scale overlay",
  "test_status": "HISTORICAL_EXPLORATORY (test previously viewed)",
  "early_valid_fraction": 0.6,
  "per_asset_early_valid_origins": 512,
  "per_asset_late_valid_origins": 512,
  "particles": 256,
  "bootstrap": 1000,
  "state_predictors": [
    "slow",
    "gap",
    "now",
    "slope15"
  ],
  "state_log_variance_clip": 0.85,
  "m2_log_variance_ratio_clip": 1.5,
  "state_l2_penalty": 0.025,
  "anchor_l2_penalty": 0.003,
  "primary_metric": "return_nll",
  "additional_variance_metric": "return_qlike (predictive net-return variance proper score)",
  "native_kernel_unchanged": true,
  "note": "No future price observations in features. Validation sample overlap with native t df selection; no untouched inference."
}


## 1. Synthetic and regression tests (no real market-data use)

In [2]:
import subprocess
p=subprocess.run([sys.executable,'-m','pytest','-q',
  str(ROOT/'tests'/'test_markov_conditional_variance.py'),
  str(ROOT/'tests'/'test_markov_return_horizon_calibration.py'),
  str(ROOT/'tests'/'test_student_t_return_markov.py'),
  str(ROOT/'tests'/'test_markov_lift_distributions.py')],
  cwd=str(ROOT),capture_output=True,text=True)
print(p.stdout[-5000:]); print(p.stderr[-1500:])
assert p.returncode==0, 'Tests must pass before market data evaluation' 

.....................                                                    [100%]
21 passed in 27.93s




## 2. Native five-minute history, seven horizon targets

Read **Binance spot** close prices only. Features are causal, training-only clock centering; missing bars and split crossing are censored. Historical data ends strictly before 2026-09-19 UTC.

In [3]:
raw,excluded=v14.load_returns(HIST,ASSETS)
panel=v14.panelize(raw)
del raw
features=v14.make_features(panel)
frame=v14.make_targets(panel,features,v17.HORIZONS)
print('Assets:', sorted(panel.underlying.unique()),'| excluded:',excluded)
print(panel.groupby('split').agg(days=('date','nunique'),rows=('return_5m','size')).to_string())
print('Elapsed',round(time.monotonic()-start,1),'s') 

Assets: ['AVAX', 'BTC', 'ETH', 'SOL', 'TRX', 'XRP'] | excluded: {'HYPE': 'insufficient history'}
       days    rows
split              
test    150  259200
train   448  774138
valid   150  259200
Elapsed 38.8 s


## 3. Fit conditional variance overlays; historical retrospective score

The native Student-t return process is refitted on TRAIN with original VD15 nu selection using VALID; VD16 base scale and shape are fitted on **early VALID** and never touched thereafter. Conditional variance corrections fit on the same early VALID subsample; late VALID is purged and used only for descriptive selection. Historical TEST scores are exploratory because the period was inspected in prior modules.

In [4]:
scores,boot,per_asset,variance_fit,params=v17.run_all(frame)
assert len(scores)==len(v17.HORIZONS)*len(v17.MODELS)
assert (scores.groupby('horizon_min').n_origins.nunique()==1).all()
print(scores.groupby('horizon_min').agg(models=('model','nunique'),origins=('n_origins','first'),days=('n_days','first')).to_string())
print('Elapsed',round(time.monotonic()-start,1),'s')

             models  origins  days
horizon_min                       
5                 6   259194   150
10                6   259188   150
15                6   259182   150
30                6   259164   150
60                6   259128   150
120               6   259056   150
240               6   258912   150
Elapsed 1214.8 s


## 4. Explainable variance fits, density scores, tail scores and variance QLIKE

A significant NLL gain is **not** enough to promote a model if tail calibration or net-return QLIKE worsens. `AM_T_VAR_ANCHOR` borrows M2's variance *forecast*, not its density; do not interpret it as purely incremental Markov information.

In [5]:
display(variance_fit.round(6))
display(scores.round(7))
display(boot[(boot.horizon_min==60)&boot.metric.isin(['nll','return_qlike','tail_logloss'])].round(7))
display(per_asset[per_asset.horizon_min==60].round(6))
print('WARNING: historical TEST has been inspected in VD13-16; no prospective confirmation.')

,horizon_min,model,cal_nll,select_nll,anchor_weight,max_abs_beta,cal_origins,select_origins,boundary
0,5,AM_T_BOTH,-5.168715,-5.204328,0.000000,0.000000,3072,3072,2026-02-21 00:00:00+00:00
1,5,AM_T_VAR_STATE,-5.194296,-5.238367,0.000000,0.213958,3072,3072,2026-02-21 00:00:00+00:00
2,5,AM_T_VAR_ANCHOR,-5.168714,-5.204326,0.000049,0.000000,3072,3072,2026-02-21 00:00:00+00:00
3,5,AM_T_VAR_COMBO,-5.194591,-5.238558,0.067905,0.204927,3072,3072,2026-02-21 00:00:00+00:00
4,10,AM_T_BOTH,-4.785222,-4.823523,0.000000,0.000000,3072,3072,2026-02-21 00:00:00+00:00
5,10,AM_T_VAR_STATE,-4.820660,-4.859243,0.000000,0.319572,3072,3072,2026-02-21 00:00:00+00:00
6,10,AM_T_VAR_ANCHOR,-4.787399,-4.827360,0.183963,0.000000,3072,3072,2026-02-21 00:00:00+00:00
7,10,AM_T_VAR_COMBO,-4.820900,-4.859656,0.275879,0.262940,3072,3072,2026-02-21 00:00:00+00:00
8,15,AM_T_BOTH,-4.614587,-4.581400,0.000000,0.000000,3072,3072,2026-02-21 00:00:00+00:00
9,15,AM_T_VAR_STATE,-4.647603,-4.612368,0.000000,0.245826,3072,3072,2026-02-21 00:00:00+00:00


,horizon_min,model,n_origins,n_days,nll,tail_logloss,tail_brier,tail_predicted,tail_observed,interval_miss95,return_qlike,pred_variance
0,5,AM_T,259194,150,-5.354158,0.078921,0.016741,0.013444,0.018048,0.105006,-12.096066,0.000002
1,5,AM_T_BOTH,259194,150,-5.399202,0.077706,0.017244,0.028590,0.018048,0.050950,-12.377464,0.000003
2,5,AM_T_VAR_STATE,259194,150,-5.420286,0.074366,0.016638,0.022800,0.018048,0.038840,-12.472015,0.000003
3,5,AM_T_VAR_ANCHOR,259194,150,-5.399202,0.077706,0.017244,0.028588,0.018048,0.050958,-12.377460,0.000003
4,5,AM_T_VAR_COMBO,259194,150,-5.420839,0.074275,0.016633,0.022582,0.018048,0.039206,-12.473892,0.000003
5,5,M2_student_t,259194,150,-5.081550,0.081984,0.016794,0.009748,0.018048,0.231525,-9.389672,0.000001
6,10,AM_T,259188,150,-5.017126,0.082234,0.016786,0.012939,0.018122,0.099704,-11.457962,0.000004
7,10,AM_T_BOTH,259188,150,-5.060350,0.079010,0.017418,0.030072,0.018122,0.046877,-11.712053,0.000007
8,10,AM_T_VAR_STATE,259188,150,-5.075224,0.074973,0.016681,0.024563,0.018122,0.036040,-11.783531,0.000006
9,10,AM_T_VAR_ANCHOR,259188,150,-5.067259,0.076816,0.017013,0.025745,0.018122,0.052325,-11.728012,0.000006


,horizon_min,comparison,metric,delta,q025,q975,test_days,origins
128,60,AM_T_BOTH - AM_T,nll,-4.928980e-02,-0.062521,-0.037132,150,259128
129,60,AM_T_BOTH - AM_T,tail_logloss,-1.021850e-02,-0.015234,-0.005813,150,259128
131,60,AM_T_BOTH - AM_T,return_qlike,-1.880296e-01,-0.244865,-0.140058,150,259128
132,60,AM_T_VAR_STATE - AM_T_BOTH,nll,-1.432410e-02,-0.017563,-0.011056,150,259128
133,60,AM_T_VAR_STATE - AM_T_BOTH,tail_logloss,-2.397700e-03,-0.003473,-0.001287,150,259128
135,60,AM_T_VAR_STATE - AM_T_BOTH,return_qlike,-7.477830e-02,-0.100319,-0.054599,150,259128
136,60,AM_T_VAR_ANCHOR - AM_T_BOTH,nll,-1.780210e-02,-0.020308,-0.015406,150,259128
137,60,AM_T_VAR_ANCHOR - AM_T_BOTH,tail_logloss,-3.004800e-03,-0.004063,-0.001973,150,259128
139,60,AM_T_VAR_ANCHOR - AM_T_BOTH,return_qlike,-7.875920e-02,-0.094884,-0.064241,150,259128
140,60,AM_T_VAR_COMBO - AM_T_BOTH,nll,-1.450470e-02,-0.018054,-0.010871,150,259128


,horizon_min,asset,model,n_origins,nll,tail_logloss,tail_brier,tail_predicted,tail_observed,interval_miss95,return_qlike,pred_variance
144,60,AVAX,AM_T,43188,-3.684984,0.102006,0.017708,0.011202,0.018547,0.086922,-8.906216,0.000038
145,60,AVAX,AM_T_BOTH,43188,-3.726023,0.089163,0.018066,0.025674,0.018547,0.047976,-9.056104,0.000059
146,60,AVAX,AM_T_VAR_STATE,43188,-3.733227,0.087960,0.017978,0.024577,0.018547,0.042489,-9.091703,0.000058
147,60,AVAX,AM_T_VAR_ANCHOR,43188,-3.741007,0.086278,0.017645,0.017127,0.018547,0.052005,-9.104130,0.000046
148,60,AVAX,AM_T_VAR_COMBO,43188,-3.734709,0.087888,0.017917,0.022467,0.018547,0.047421,-9.086322,0.000054
149,60,AVAX,M2_student_t,43188,-3.741882,0.085510,0.017665,0.015438,0.018547,0.059646,-9.103752,0.000044
150,60,BTC,AM_T,43188,-4.284745,0.132675,0.029088,0.024247,0.031467,0.083287,-10.164263,0.000014
151,60,BTC,AM_T_BOTH,43188,-4.324238,0.125297,0.029693,0.045472,0.031467,0.046819,-10.279351,0.000022
152,60,BTC,AM_T_VAR_STATE,43188,-4.340274,0.120733,0.028940,0.036675,0.031467,0.049065,-10.347310,0.000018
153,60,BTC,AM_T_VAR_ANCHOR,43188,-4.342385,0.120118,0.028718,0.029875,0.031467,0.052978,-10.344549,0.000016


## 5. Numerical QMC convergence on late VALID, not TEST

Check nested 256 and 512 Sobol nodes at 5m, 1h, 4h. The size of QMC-induced NLL changes bounds what small forecasting improvements can mean.

In [6]:
stability=v17.particle_sensitivity_validation(frame,params['native_kernels'],
  params['markov_transitions'],params['calibrations'],horizons=(5,60,240))
display(stability.round(7))

,horizon_min,asset,model,origins,particles_low,particles_high,nll_low,nll_high,high_minus_low
0,5,AVAX,AM_T_BOTH,96,256,512,-4.805749,-4.802103,0.003646
1,5,AVAX,AM_T_VAR_STATE,96,256,512,-4.837544,-4.834669,0.002876
2,5,AVAX,AM_T_VAR_ANCHOR,96,256,512,-4.805743,-4.802096,0.003647
3,5,AVAX,AM_T_VAR_COMBO,96,256,512,-4.836937,-4.833844,0.003092
4,5,BTC,AM_T_BOTH,96,256,512,-5.245401,-5.245464,-0.000063
...,...,...,...,...,...,...,...,...,...
67,240,TRX,AM_T_VAR_COMBO,96,256,512,-4.067021,-4.067360,-0.000339
68,240,XRP,AM_T_BOTH,96,256,512,-3.183658,-3.182415,0.001242
69,240,XRP,AM_T_VAR_STATE,96,256,512,-3.202159,-3.200798,0.001360
70,240,XRP,AM_T_VAR_ANCHOR,96,256,512,-3.204260,-3.203953,0.000307


## 6. Exploratory decision and compact saved outputs

In [7]:
decision=v17.decide(scores,boot,params)
assert decision['status']=='HISTORICAL_EXPLORATORY'
assert not decision['historical_confirmation_permitted']
print(json.dumps(decision,indent=2))
scores.to_csv(OUT/'17_scores.csv',index=False,float_format='%.10g')
boot.to_csv(OUT/'17_bootstrap.csv',index=False,float_format='%.10g')
per_asset.to_csv(OUT/'17_per_asset.csv',index=False,float_format='%.10g')
variance_fit.to_csv(OUT/'17_variance_fit_grid.csv',index=False,float_format='%.10g')
stability.to_csv(OUT/'17_particle_sensitivity.csv',index=False,float_format='%.10g')
(OUT/'17_decision.json').write_text(json.dumps(decision,indent=2),encoding='utf-8')
manifest={'module':'Volatility_Dynamics_17','version':v17.VERSION,'registration':v17.registration(),
  'source':str(HIST),'sample_first':str(panel.state_time.min()),'sample_last':str(panel.state_time.max()),
  'assets':sorted(panel.underlying.unique().tolist()),'excluded_assets':excluded,
  'days_by_split':panel.groupby('split').date.nunique().to_dict(),
  'fitted_parameters':params,'decision':decision,
  'warning':'Historical period already inspected; conditional forecasting overlays not a path-consistent Markov process.'}
(OUT/'17_manifest.json').write_text(json.dumps(manifest,indent=2,default=str),encoding='utf-8')
print('Saved 7 compact research artifacts:',OUT)
print('No raw returns, forecast-origin dumps, DuckDB or collector writes.')

{
  "status": "HISTORICAL_EXPLORATORY",
  "primary_horizon_min": 60,
  "test_days": 150,
  "primary_flags": {
    "STATE_beats_BASE": true,
    "ANCHOR_beats_BASE": true,
    "COMBO_beats_BASE": true,
    "COMBO_beats_M2": false
  },
  "late_valid_selected": {
    "5": "AM_T_VAR_COMBO",
    "10": "AM_T_VAR_COMBO",
    "15": "AM_T_VAR_STATE",
    "30": "AM_T_VAR_STATE",
    "60": "AM_T_VAR_STATE",
    "120": "AM_T_VAR_STATE",
    "240": "AM_T_VAR_STATE"
  },
  "historical_confirmation_permitted": false,
  "trading_approved": false
}
Saved 7 compact research artifacts: D:\Projects\deribit_market_data\notebooks\Volatility_Dynamics\17
No raw returns, forecast-origin dumps, DuckDB or collector writes.
